# Measure tissue thickness

Variable tissue thickness means a fraction of every FOV's z-stack has no real tissue signal at all -- wasted imaging time. This notebook maps, per FOV, where (in z, um) real tissue signal ends, using one already-finished round (default: `cells`), then estimates the time/data savings from trimming future rounds to that depth plus a small margin.

Part of the `lineage_tracing/merfish_multi_z` prepare-imaging sequence: run after notebook 01 (cells hal_config) and notebook 03 (positions), against the finished `cells` round -- notebook 05 reads this notebook's exported table to decide each FOV's z-depth tier for the bits rounds. Follows the architecture rules in [`NOTEBOOK_GUIDELINES.md`](../../../../NOTEBOOK_GUIDELINES.md) (repo root) throughout -- every nontrivial step below is a **calculation cell** (cached under `analysis/cache/measure_tissue_thickness/`, with `ProgressReporter` progress, skipping recomputation when its cache still matches the current inputs) followed by a separate **display cell** (plot or printed summary).

Deliberately a **lean, production version** of `notebooks/misc/measure_tissue_thickness_test.ipynb` (the full exploratory/R&D notebook, with per-FOV texture profiles, mosaics, GIFs, and other diagnostics) -- only what notebook 05 actually needs: the NTP-based z_last calculation, a fixed margin, a heatmap of the result, and the theoretical time/data savings estimate.

Procedure:
1. Resolve the target round's frame table and `CHANNEL_NM`'s z-steps.
2. For every FOV, read every z-plane of `CHANNEL_NM` (still far fewer than the round's full multi-color frame count) and build an EXACT, bin-width-1 histogram of each frame -- a true Counter over observed pixel intensities (`analysis.fov.compute_channel_counters`, stored sparsely via `numpy.unique`, cached per FOV). This is the heaviest read step in the notebook -- it has a **SLURM array option** (`USE_SLURM_ARRAY` in section 4), per the standing convention that any heavy multi-FOV I/O task should offer one.
3. Across every FOV and z, find the frame with the **highest mean intensity** (a visual "what does real tissue look like" reference) and the `N_BACKGROUND_FRAMES` frames with the **lowest mean intensity** (the best available proxy for pure background/no-tissue signal). Display both, overlay all the background frames' histograms plus the tissue frame's histogram, and derive `THRESHOLD` as the highest pixel value observed among those background frames. Review the plot and override `THRESHOLD` manually if it looks wrong.
4. For every FOV, derive its true-pixel-count (NTP) profile directly from its cached Counter (no further disk read), reporting `z_last_um` (the deepest z with signal above `NTP_THRESHOLD`).
5. Lay every FOV's `z_last_um` out on its stage-position grid and plot as a heatmap.
6. Add a small margin (`Z_MARGIN_UM`, default 1 um) to every FOV's `z_last_um` and estimate the time/data savings from trimming future rounds to that depth, using the **theoretical** per-frame time (from this round's HAL `<exposure_time>`, the same convention `acquisition.dave.estimate_dave_experiment` uses) -- the only rate available here, since the bits rounds this is planning for haven't been imaged yet.
7. Export the per-FOV z table notebook 05 needs.


## 1 — Setup

In [ ]:
import os
import sys
import json
import csv
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/prepare_imaging/<variant>/<acquisition>/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config      import ExperimentConfig
from MERci.common.metadata    import ExperimentMetadata
from MERci.progress           import ProgressTracker
from MERci.progress_display   import ProgressReporter, format_duration
from MERci.common.io          import iter_image_frames
from MERci.analysis.fov       import (
    compute_channel_counters, save_channel_counters, load_channel_counters,
    counter_mean, counter_percentile, rebin_counter, ntp_profile_from_counters,
)
from MERci.acquisition.configs import find_frame_table_for_hal_config, read_hal_exposure_time

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME  = SAMPLE_DIR.name
IMAGE_SUFFIX = ".zarr"   # must match what HAL wrote

# Which microscope this experiment was acquired on.
MICROSCOPE = "ST2"

# Which round to use -- by imaging_type (default "cells"), or set ROUND_ID directly to override.
ROUND_IMAGING_TYPE = "cells"
ROUND_ID            = None

# Channel to measure tissue depth for.
CHANNEL_NM = 405.0

# A z-plane still counts as "has tissue" if its true-pixel count (NTP) exceeds this.
NTP_THRESHOLD = 1

# Heatmap color scale upper bound (micron).
MAX_Z_COLORMAP = 70.0

# Display-histogram resolution (section 5) -- these are re-binned on demand from the
# exact per-intensity Counter, so changing this never requires recomputing anything.
DISPLAY_HIST_BINS      = 200
LINEAR_HIST_PERCENTILE = 99.0   # upper bound of the linear-scale display histogram

# How many of the lowest-mean-intensity frames (across every FOV/z) to treat as
# "confidently background" for threshold estimation (section 5), and what
# percentile of each background frame's own pixel distribution to take as its
# noise ceiling (100 = literal max pixel value observed in that frame).
N_BACKGROUND_FRAMES  = 10
BACKGROUND_PERCENTILE = 100.0

# Binarization intensity threshold; None = auto-estimate as the highest pixel
# value observed across the N_BACKGROUND_FRAMES lowest-mean frames (section 5)
# -- review that plot before trusting the estimate on a new experiment.
THRESHOLD = None

# Explicit plot font sizes (NOTEBOOK_GUIDELINES.md #5) -- matplotlib's default
# sizes shrink relative to figsize, so a wide/short figure reads noticeably
# smaller than a square one at the same nominal size.
PLOT_TITLE_FONTSIZE    = 14
PLOT_LABEL_FONTSIZE    = 12
PLOT_TICK_FONTSIZE     = 11
PLOT_LEGEND_FONTSIZE   = 10
PLOT_SUPTITLE_FONTSIZE = 15

print(f"Sample name        : {SAMPLE_NAME}")
print(f"Microscope         : {MICROSCOPE}")
print(f"Round imaging type : {ROUND_IMAGING_TYPE}  (ROUND_ID override: {ROUND_ID})")
print(f"Channel            : {CHANNEL_NM} nm")

In [ ]:
config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{SAMPLE_NAME}.txt",
    image_suffix   = IMAGE_SUFFIX,
    microscope     = MICROSCOPE,
)

meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)
tracker = ProgressTracker(config.analysis_dir)

figures_dir = config.analysis_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

# NOTEBOOK_GUIDELINES.md #2/#3: every calculation cell below caches its result
# under analysis/cache/<notebook_name>/ and skips recomputation when a valid
# cache is already there.
NOTEBOOK_NAME = "measure_tissue_thickness"
cache_dir     = config.analysis_dir / "cache" / NOTEBOOK_NAME
channel_counters_dir = cache_dir / "channel_counters"
channel_counters_dir.mkdir(parents=True, exist_ok=True)

print(f"Rounds : {meta.n_rounds}")
print(f"FOVs   : {meta.n_fovs}")
print(f"Figures: {figures_dir}")
print(f"Cache  : {cache_dir}")

## 3 — Resolve the target round and its frame table

In [ ]:
def resolve_round_id(meta, imaging_type):
    """First round_id whose series carry the given imaging_type."""
    for rid in meta.valid_round_ids():
        if any((s.imaging_type or "").strip().lower() == imaging_type.strip().lower()
               for s in meta.series_for_round(rid)):
            return rid
    raise ValueError(f"No round found with imaging_type={imaging_type!r}")


def load_round_frame_table(round_id, config, meta):
    """Frame table (columns color/channel/z, 0-based frame-index rows) for round_id's HAL config."""
    for s in meta.series_for_round(round_id):
        if not s.hal_config:
            continue
        hal_path = Path(config.settings_dir) / s.hal_config
        ft_path  = find_frame_table_for_hal_config(hal_path, config.metadata_dir)
        if ft_path and ft_path.exists():
            return pd.read_csv(ft_path, index_col=0)
    raise FileNotFoundError(f"No frame table found for round {round_id}")


target_round_id = ROUND_ID if ROUND_ID is not None else resolve_round_id(meta, ROUND_IMAGING_TYPE)
if not meta.round_fully_written(target_round_id):
    print(f"WARNING: round {target_round_id} is not yet fully written on disk -- "
          f"results below will be based on a partial FOV set.")

frame_table = load_round_frame_table(target_round_id, config, meta)

channel_frames = (
    frame_table[frame_table["color"].round(0) == round(CHANNEL_NM)]
    .sort_values("z")
)
if channel_frames.empty:
    raise ValueError(f"No frames found for channel {CHANNEL_NM} nm in round {target_round_id}'s frame table.")

z_frame_indices = list(zip(channel_frames.index.tolist(), channel_frames["z"].tolist()))

print(f"Target round : {target_round_id}")
print(f"Channel {CHANNEL_NM} nm has {len(z_frame_indices)} z-step(s) in this round's frame table.")

## 4 — Compute (or load cached) exact per-z Counter histograms for `CHANNEL_NM`

For every FOV, reads every z-plane of `CHANNEL_NM` (still far fewer than the round's
full multi-color frame count) and builds an EXACT, bin-width-1 histogram of each
frame -- a true Counter over observed pixel intensities (`analysis.fov.
compute_channel_counters`, stored sparsely: only intensity values that actually
occur, via `numpy.unique`). This is the reference cell for `NOTEBOOK_GUIDELINES.md`
#2-4: per-FOV caching (only what's actually missing gets computed), and
`ProgressReporter`-driven progress on the remaining work.

**This is the heaviest read step in the notebook** (a full z-sweep of one channel,
per FOV) -- `USE_SLURM_ARRAY` below submits one array task per still-missing FOV
(`build_channel_counters_array_script` + `cli_compute_channel_counters.py`) instead
of computing sequentially, per the standing convention that any heavy multi-FOV I/O
task in these notebooks should offer a SLURM option.

In [ ]:
def channel_counters_path(fpath):
    return channel_counters_dir / f"{Path(fpath).stem}_counters.npz"


files = meta.files_for_round(target_round_id)
print(f"Round {target_round_id}: {len(files)} FOV file(s) expected.")

channel_counters = {}   # fov_id -> compute_channel_counters()-shaped dict
from_own_cache, to_compute = [], []
n_missing_on_disk = 0

for fpath in files:
    if channel_counters_path(fpath).exists():
        from_own_cache.append(fpath)
    elif fpath.exists():
        to_compute.append(fpath)
    else:
        n_missing_on_disk += 1

for fpath in from_own_cache:
    channel_counters[meta.fov_id_of_file(fpath)] = load_channel_counters(channel_counters_path(fpath))
print(f"{len(from_own_cache)} channel Counter(s) already cached -- loaded directly.")
print(f"{len(to_compute)} FOV(s) need their channel Counter computed "
      f"({n_missing_on_disk} not yet written on disk).")

# ---- Optional: submit a SLURM array job instead of computing locally -----
USE_SLURM_ARRAY         = False   # set True on a cluster login node
SLURM_ARRAY_CONCURRENCY = 50
SLURM_MEM               = "4gb"
SLURM_TIME              = "00:20:00"

n_computed = 0
if to_compute and USE_SLURM_ARRAY:
    from MERci.acquisition.cluster_submit import (
        build_channel_counters_array_script, submit_sbatch, is_job_active,
    )

    counters_job_sentinel = cache_dir / f"channel_counters_job_round{target_round_id}.json"
    cached_job = json.loads(counters_job_sentinel.read_text()) if counters_job_sentinel.exists() else None

    if (cached_job is not None and cached_job.get("n_pending") == len(to_compute)
            and is_job_active(cached_job["job_id"])):
        print(f"SLURM array job {cached_job['job_id']} is still active "
              f"({len(to_compute)} FOV(s) pending) -- re-run this cell later once it finishes.")
    else:
        manifest_path = cache_dir / f"channel_counters_manifest_round{target_round_id}.txt"
        manifest_path.write_text("\n".join(str(fpath) for fpath in to_compute) + "\n")

        frame_idx_list = [idx for idx, _ in z_frame_indices]
        z_um_list      = [z for _, z in z_frame_indices]
        script_path    = cache_dir / f"channel_counters_round{target_round_id}.sh"
        build_channel_counters_array_script(
            sample_dir=SAMPLE_DIR, manifest_path=manifest_path, output_dir=channel_counters_dir,
            frame_indices=frame_idx_list, z_um_values=z_um_list,
            n_pending=len(to_compute), output_path=script_path,
            array_concurrency=SLURM_ARRAY_CONCURRENCY, mem=SLURM_MEM, time=SLURM_TIME,
        )
        job_id = submit_sbatch(script_path)
        if job_id is not None:
            counters_job_sentinel.write_text(json.dumps({"job_id": job_id, "n_pending": len(to_compute)}))
            print(f"Submitted SLURM array job {job_id} for {len(to_compute)} FOV(s) -- "
                  f"re-run this cell later once it finishes to load the results.")
        else:
            print("sbatch submission failed (see the logged error above) -- fix the issue and re-run this cell.")
elif to_compute:
    reporter = ProgressReporter(total=len(to_compute), label="Computing channel Counters")
    for fpath in reporter.wrap(to_compute):
        counters = compute_channel_counters(
            fpath, z_frame_indices,
            frame_width=config.frame_width, frame_height=config.frame_height,
        )
        save_channel_counters(channel_counters_path(fpath), counters)
        channel_counters[meta.fov_id_of_file(fpath)] = counters
        n_computed += 1

print(f"Channel Counters ready for {len(channel_counters)} / {len(files)} FOVs "
      f"({n_computed} newly computed this run, {n_missing_on_disk} not yet written on disk).")

## 5 — Reference frames: highest-mean (tissue) + lowest-mean (background) frames

**Calculation cell**, then **display cell** (`NOTEBOOK_GUIDELINES.md` #1).

Across every FOV and z, finds the frame with the highest mean intensity (a
visual "what does real tissue look like" reference) and the
`N_BACKGROUND_FRAMES` frames with the lowest mean intensity (the best
available proxy for pure background/no-tissue signal).

`THRESHOLD` is auto-estimated as the highest pixel value observed among
those `N_BACKGROUND_FRAMES` background frames -- the highest pixel value
that can plausibly occur as background noise, derived only from frames
confidently known to be background. Review the overlaid histogram plot
before trusting the estimate -- if it looks wrong, set `THRESHOLD` by hand
in section 2 and re-run from here.

In [ ]:
# ---- Calculation --------------------------------------------------------
reference_frames_cache = cache_dir / f"reference_frames_round{target_round_id}.npz"
n_frames_now = sum(len(c["values_per_z"]) for c in channel_counters.values())

cached = None
if reference_frames_cache.exists():
    cached = np.load(reference_frames_cache)
    if int(cached["n_frames_considered"]) != n_frames_now:
        print(f"Cached reference-frame selection is stale "
              f"({int(cached['n_frames_considered'])} vs. {n_frames_now} frame(s) now available) -- recomputing.")
        cached = None

if cached is not None:
    n_frames_considered = int(cached["n_frames_considered"])
    best_mean, best_fov_id, best_pos, best_frame_idx, best_z_um = (
        float(cached["best_mean"]), int(cached["best_fov_id"]), int(cached["best_pos"]),
        int(cached["best_frame_idx"]), float(cached["best_z_um"]),
    )
    worst_fov_ids, worst_pos, worst_frame_idx, worst_z_um = (
        cached["worst_fov_ids"], cached["worst_pos"], cached["worst_frame_idx"], cached["worst_z_um"],
    )
    print(f"Loaded cached reference-frame selection ({reference_frames_cache.name}) -- "
          f"{n_frames_considered} FOV x z combination(s), matches current data.")
else:
    frame_records = []   # (mean, fov_id, pos_in_z, frame_idx, z_um)
    reporter = ProgressReporter(total=len(channel_counters), label="Scanning frame means")
    for fov_id, counters in reporter.wrap(channel_counters.items()):
        for pos, (values, counts) in enumerate(zip(counters["values_per_z"], counters["counts_per_z"])):
            mean = counter_mean(values, counts)
            frame_records.append((mean, fov_id, pos, int(counters["frame_indices"][pos]), float(counters["z_um"][pos])))

    n_frames_considered = len(frame_records)
    frame_records.sort(key=lambda r: r[0])
    best_mean, best_fov_id, best_pos, best_frame_idx, best_z_um = frame_records[-1]
    worst_records = frame_records[:N_BACKGROUND_FRAMES]

    worst_fov_ids   = np.array([r[1] for r in worst_records], dtype=np.int64)
    worst_pos       = np.array([r[2] for r in worst_records], dtype=np.int64)
    worst_frame_idx = np.array([r[3] for r in worst_records], dtype=np.int64)
    worst_z_um      = np.array([r[4] for r in worst_records], dtype=np.float64)

    np.savez_compressed(
        reference_frames_cache,
        n_frames_considered=n_frames_considered,
        best_mean=best_mean, best_fov_id=best_fov_id, best_pos=best_pos,
        best_frame_idx=best_frame_idx, best_z_um=best_z_um,
        worst_fov_ids=worst_fov_ids, worst_pos=worst_pos,
        worst_frame_idx=worst_frame_idx, worst_z_um=worst_z_um,
    )
    print(f"Scanned {n_frames_considered} FOV x z combination(s); cached selection to {reference_frames_cache.name}.")

best_values, best_counts = (channel_counters[best_fov_id]["values_per_z"][best_pos],
                            channel_counters[best_fov_id]["counts_per_z"][best_pos])
worst_value_counts = [
    (channel_counters[int(fov_id)]["values_per_z"][int(pos)], channel_counters[int(fov_id)]["counts_per_z"][int(pos)])
    for fov_id, pos in zip(worst_fov_ids, worst_pos)
]

estimated_threshold = float(max(
    counter_percentile(values, counts, BACKGROUND_PERCENTILE) for values, counts in worst_value_counts
))

print(f"Highest-mean frame: FOV {best_fov_id}, frame_idx={best_frame_idx}, z={best_z_um:.2f} um (mean {best_mean:.0f})")
print(f"Lowest-mean {len(worst_fov_ids)} frame(s) (background reference):")
for fov_id, frame_idx, z_um in zip(worst_fov_ids, worst_frame_idx, worst_z_um):
    print(f"  FOV {fov_id}, frame_idx={frame_idx}, z={z_um:.2f} um")
print(f"Estimated background noise ceiling (p{BACKGROUND_PERCENTILE:.1f} of {len(worst_value_counts)} "
      f"background frame(s)): {estimated_threshold:.0f}")

In [ ]:
# ---- Display --------------------------------------------------------------
# Counters have no spatial information -- re-read the tissue reference frame
# and the single emptiest background frame to display them.
best_fpath  = next(f for f in files if meta.fov_id_of_file(f) == best_fov_id)
worst_fpath = next(f for f in files if meta.fov_id_of_file(f) == int(worst_fov_ids[0]))
best_frame  = next(frame for _, frame in iter_image_frames(
    best_fpath, [best_frame_idx], frame_width=config.frame_width, frame_height=config.frame_height,
))
worst_frame = next(frame for _, frame in iter_image_frames(
    worst_fpath, [int(worst_frame_idx[0])], frame_width=config.frame_width, frame_height=config.frame_height,
))

fig_img, axes_img = plt.subplots(1, 2, figsize=(10, 5))
for ax, frame, fov_id, z_um, title in zip(
    axes_img, (worst_frame, best_frame), (int(worst_fov_ids[0]), best_fov_id), (float(worst_z_um[0]), best_z_um),
    ("Lowest-mean frame (background)", "Highest-mean frame (tissue)"),
):
    im = ax.imshow(frame, cmap="gray")
    ax.set_title(f"{title}\nFOV {fov_id}, z={z_um:.1f} um", fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    cbar = fig_img.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
fig_img.tight_layout()
fig_img.savefig(figures_dir / f"tissue_thickness_reference_frames_round{target_round_id}.png", dpi=150)
plt.show()

# Overlay every background frame's histogram (thin lines) + the tissue
# reference frame's histogram (bold). Log-scale (full range) and
# linear-scale (combined min -> the tissue frame's LINEAR_HIST_PERCENTILE),
# both re-binned on demand from the exact Counters -- no raw pixel re-read.
combined_min = float(min(min(v.min() for v, _ in worst_value_counts), best_values.min()))
combined_max = float(max(max(v.max() for v, _ in worst_value_counts), best_values.max()))
pct_value    = counter_percentile(best_values, best_counts, LINEAR_HIST_PERCENTILE)

log_edges       = np.logspace(np.log10(max(combined_min, 1)), np.log10(combined_max), DISPLAY_HIST_BINS + 1)
log_bin_centers = np.sqrt(log_edges[:-1] * log_edges[1:])   # geometric mean = correct center in log space

linear_edges       = np.linspace(combined_min, max(pct_value, combined_min + 1), DISPLAY_HIST_BINS + 1)
linear_bin_centers = 0.5 * (linear_edges[:-1] + linear_edges[1:])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for values, counts in worst_value_counts:
    axes[0].plot(log_bin_centers, rebin_counter(values, counts, log_edges), "-", color="steelblue", alpha=0.5, lw=1.0)
    axes[1].plot(linear_bin_centers, rebin_counter(values, counts, linear_edges), "-", color="steelblue", alpha=0.5, lw=1.0)
axes[0].plot(log_bin_centers, rebin_counter(best_values, best_counts, log_edges), "-", color="darkorange", lw=1.8)
axes[1].plot(linear_bin_centers, rebin_counter(best_values, best_counts, linear_edges), "-", color="darkorange", lw=1.8)
for ax in axes:
    ax.plot([], [], "-", color="steelblue", alpha=0.5, lw=1.0,
            label=f"{len(worst_value_counts)} lowest-mean (background) frames")
    ax.plot([], [], "-", color="darkorange", lw=1.8, label=f"highest-mean frame (FOV {best_fov_id})")

axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel(f"Intensity  (channel {CHANNEL_NM:.0f} nm)", fontsize=PLOT_LABEL_FONTSIZE)
axes[0].set_ylabel("Pixel count", fontsize=PLOT_LABEL_FONTSIZE)
axes[0].set_title("Log-scale (full range)", fontsize=PLOT_TITLE_FONTSIZE)

axes[1].set_xlabel(f"Intensity  (channel {CHANNEL_NM:.0f} nm)", fontsize=PLOT_LABEL_FONTSIZE)
axes[1].set_ylabel("Pixel count", fontsize=PLOT_LABEL_FONTSIZE)
axes[1].set_title(f"Linear-scale (min={combined_min:.0f} -> p{LINEAR_HIST_PERCENTILE:.0f}={pct_value:.0f})",
                   fontsize=PLOT_TITLE_FONTSIZE)

for ax in axes:
    ax.axvline(estimated_threshold, color="crimson", linestyle="--", lw=1.5,
               label=f"background noise ceiling = {estimated_threshold:.0f}")
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)

fig.suptitle(f"Round {target_round_id} -- {len(worst_value_counts)} lowest-mean (background) vs. "
             f"highest-mean (tissue) frame histograms", fontsize=PLOT_SUPTITLE_FONTSIZE)
fig.tight_layout()
fig.savefig(figures_dir / f"tissue_thickness_histogram_round{target_round_id}.png", dpi=150)
plt.show()

if THRESHOLD is None:
    THRESHOLD = estimated_threshold
print(f"Using THRESHOLD = {THRESHOLD:.0f}")

## 6 — Per-FOV: true-pixel-count (NTP) profile, derived from cached Counters

**Calculation cell**, then **display cell** (`NOTEBOOK_GUIDELINES.md` #1).

Purely in-memory: every FOV's exact per-z Counter is already cached/loaded from
section 4, so deriving each z's true-pixel count against `THRESHOLD`
(`analysis.fov.ntp_profile_from_counters`) needs no further disk read at all.
Reports `z_last_um` (deepest z with signal), the value section 7/8 use.

In [ ]:
# ---- Calculation --------------------------------------------------------
results_cache      = cache_dir / f"ntp_profile_round{target_round_id}.csv"
results_meta_cache = cache_dir / f"ntp_profile_round{target_round_id}.json"
cache_signature     = {"threshold": float(THRESHOLD), "ntp_threshold": float(NTP_THRESHOLD),
                        "n_fovs": len(channel_counters)}

cached_signature = json.loads(results_meta_cache.read_text()) if results_meta_cache.exists() else None

if cached_signature == cache_signature and results_cache.exists():
    results_df = pd.read_csv(results_cache)
    print(f"Loaded cached NTP profile ({results_cache.name}) -- "
          f"THRESHOLD/NTP_THRESHOLD/FOV count unchanged since it was written.")
else:
    results = []
    reporter = ProgressReporter(total=len(channel_counters), label="Deriving NTP profiles")
    for fov_id, counters in reporter.wrap(channel_counters.items()):
        profile = ntp_profile_from_counters(counters, THRESHOLD, NTP_THRESHOLD)
        results.append({
            "fov_id":        fov_id,
            "z_first_um":    profile["z_first_um"],
            "z_last_um":     profile["z_last_um"],
            "is_contiguous": profile["is_contiguous"],
            "x_um":          meta.fovs[fov_id].position[0],
            "y_um":          meta.fovs[fov_id].position[1],
        })
    results_df = pd.DataFrame(results)
    results_df.to_csv(results_cache, index=False)
    results_meta_cache.write_text(json.dumps(cache_signature))
    print(f"Derived NTP profile for {len(results_df)} FOV(s); cached to {results_cache.name}.")

In [ ]:
# ---- Display --------------------------------------------------------------
n_no_signal      = results_df["z_last_um"].isna().sum()
n_not_contiguous = (~results_df["is_contiguous"]).sum()
print(f"{len(results_df)} FOV(s) measured; {n_no_signal} had no z-plane above NTP_THRESHOLD at all; "
      f"{n_not_contiguous} had signal turn off and back on somewhere in between (is_contiguous=False).")
print("z_last_um:")
print(results_df["z_last_um"].describe())

## 7 — Tissue-depth heatmap across the FOV grid

**Calculation cell**, then **display cell** (`NOTEBOOK_GUIDELINES.md` #1).

`z_last_um` laid out on the FOV stage grid -- only `z_last_um` (not `z_first_um`
too, unlike the full exploratory notebook), since that's the value sections 8/9
actually use.

In [ ]:
# ---- Calculation --------------------------------------------------------
def positions_to_grid_indices(fov_ids, meta):
    """Stage (x, y) positions -> integer (x_idx, y_idx) grid indices (round to the
    nearest integer micron, then rank each axis's unique values -- robust to float
    imprecision on a regular grid)."""
    xs = np.array([round(meta.fovs[f].position[0]) for f in fov_ids])
    ys = np.array([round(meta.fovs[f].position[1]) for f in fov_ids])
    unique_xs = np.sort(np.unique(xs))
    unique_ys = np.sort(np.unique(ys))
    x_rank = {v: i for i, v in enumerate(unique_xs)}
    y_rank = {v: i for i, v in enumerate(unique_ys)}
    return {f: (x_rank[xs[i]], y_rank[ys[i]]) for i, f in enumerate(fov_ids)}


def build_matrix(column, results_df, grid, n_x, n_y):
    matrix = np.full((n_y, n_x), np.nan)
    for _, row in results_df.iterrows():
        xi, yi = grid[row["fov_id"]]
        if pd.notna(row[column]):
            matrix[yi, xi] = row[column]
    return matrix


fov_ids = results_df["fov_id"].tolist()
grid    = positions_to_grid_indices(fov_ids, meta)
n_x     = max(xi for xi, _ in grid.values()) + 1
n_y     = max(yi for _, yi in grid.values()) + 1

z_last_matrix = build_matrix("z_last_um", results_df, grid, n_x, n_y)

print(f"FOV grid: {n_x} x {n_y} (columns x rows), {len(fov_ids)} FOV(s) placed.")

In [ ]:
# ---- Display --------------------------------------------------------------
fig, ax = plt.subplots(figsize=(max(6, n_x * 0.6 + 2), max(4, n_y * 0.4 + 1.5)))
im = ax.imshow(z_last_matrix, cmap="viridis", origin="upper", vmin=0, vmax=MAX_Z_COLORMAP)
ax.set_title("z_last -- signal ends", fontsize=PLOT_TITLE_FONTSIZE)
ax.set_xlabel("X grid index  (increasing stage X ->)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_ylabel("Y grid index  (increasing stage Y down)", fontsize=PLOT_LABEL_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("z (um)", fontsize=PLOT_LABEL_FONTSIZE)
cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
fig.suptitle(f"Round {target_round_id} -- tissue depth map ({len(fov_ids)} FOVs)", fontsize=PLOT_SUPTITLE_FONTSIZE)
fig.tight_layout()

fig.savefig(figures_dir / f"tissue_thickness_heatmap_round{target_round_id}.png", dpi=150)
plt.show()

results_csv = config.analysis_dir / f"tissue_thickness_round{target_round_id}.csv"
results_df.to_csv(results_csv, index=False)
print(f"Saved: {results_csv}")

## 8 — Margin + theoretical time/data savings

**Calculation cell**, then **display cell** (`NOTEBOOK_GUIDELINES.md` #1).

If a future acquisition only imaged, per FOV, up to `min(z_last_um + Z_MARGIN_UM,
Z_MAX_TRIMMED_UM)` instead of this round's full z-range, how much less disk space
and acquisition time would the round take? Uses `theoretical_time_per_frame_s`
(from this round's HAL `<exposure_time>`, the same convention
`acquisition.dave.estimate_dave_experiment` uses) -- the only rate available here,
since the bits rounds this is planning for haven't actually been imaged yet (unlike
`measure_tissue_thickness_test.ipynb`'s section 8, which also reports an
**experimental** rate measured from the already-finished cells round's own
file-write timestamps).

`Z_MARGIN_UM` defaults to **1 um** -- a small, deliberately conservative buffer
beyond each FOV's own measured signal. `Z_MAX_TRIMMED_UM` is an absolute cap,
auto-derived from this round's own deepest measured `z_last_um` + `Z_MARGIN_UM`
so it can never truncate below what was actually measured; the calculation cell
warns explicitly if you override it to something that binds below any FOV's real
signal.

In [ ]:
# Extra margin (um) kept beyond each FOV's measured z_last_um.
Z_MARGIN_UM = 1.0

# Absolute cap (um) on the trimmed depth, regardless of z_last_um. Defaults to
# this round's own deepest measured z_last_um + Z_MARGIN_UM -- i.e. by default
# the cap can NEVER truncate below what was actually measured for any FOV.
_z_last_um_max = results_df["z_last_um"].max()
Z_MAX_TRIMMED_UM = (
    float(_z_last_um_max + Z_MARGIN_UM) if pd.notna(_z_last_um_max)
    else float(frame_table["z"].max())
)

# How many rounds of the WHOLE experiment are assumed to share this round's
# z-sweep depth/configuration -- 1 = this round's own savings only.
N_ROUNDS_LIKE_THIS = 1

print(f"Z_MARGIN_UM={Z_MARGIN_UM}, Z_MAX_TRIMMED_UM={Z_MAX_TRIMMED_UM:.1f} "
      f"(auto-derived from this round's own data -- override above for a different cap), "
      f"N_ROUNDS_LIKE_THIS={N_ROUNDS_LIKE_THIS}")

In [ ]:
# ---- Calculation --------------------------------------------------------
trim_cache      = cache_dir / f"zrange_trim_round{target_round_id}.csv"
trim_meta_cache = cache_dir / f"zrange_trim_round{target_round_id}.json"
trim_signature  = {
    "z_margin_um": Z_MARGIN_UM, "z_max_trimmed_um": Z_MAX_TRIMMED_UM,
    "threshold": float(THRESHOLD), "ntp_threshold": float(NTP_THRESHOLD),
    "n_fovs": len(results_df),
}

cached_trim_signature = json.loads(trim_meta_cache.read_text()) if trim_meta_cache.exists() else None

if cached_trim_signature == trim_signature and trim_cache.exists():
    trim_df = pd.read_csv(trim_cache)
    print(f"Loaded cached trim table ({trim_cache.name}) -- trim parameters/THRESHOLD/FOV count unchanged.")
else:
    # Every color group in frame_table, keyed by its (rounded) color -- a group
    # "is z-swept" if its frames actually span more than one z value (a real
    # focus sweep); everything else is a fixed, unaffected frame.
    color_key = frame_table["color"].round(0)
    color_key = color_key.where(color_key.notna(), -1)

    zswept_groups = {}
    n_fixed_frames = 0
    for key, grp in frame_table.groupby(color_key):
        z_vals = grp["z"].to_numpy()
        if pd.Series(z_vals).nunique() > 1:
            zswept_groups[key] = z_vals
        else:
            n_fixed_frames += len(grp)

    n_zswept_frames = sum(len(v) for v in zswept_groups.values())
    print(f"Frame table: {len(frame_table)} frame(s)/FOV total -- {len(zswept_groups)} z-swept color "
          f"group(s) ({n_zswept_frames} frame(s)), {n_fixed_frames} fixed frame(s) unaffected by trimming.")

    def frames_kept_for_fov(z_needed):
        """Frame count kept under the trimmed scheme: every fixed frame, plus every
        z-swept-group frame at or below z_needed (0 z-swept frames if z_needed is None)."""
        if z_needed is None:
            return n_fixed_frames
        return n_fixed_frames + sum(int((z_vals <= z_needed).sum()) for z_vals in zswept_groups.values())

    trim_rows = []
    reporter = ProgressReporter(total=len(results_df), label="Computing per-FOV trim")
    for _, row in reporter.wrap(list(results_df.iterrows())):
        z_last   = row["z_last_um"]
        z_needed = min(z_last + Z_MARGIN_UM, Z_MAX_TRIMMED_UM) if pd.notna(z_last) else None
        n_trimmed = frames_kept_for_fov(z_needed)
        trim_rows.append({
            "fov_id":            row["fov_id"],
            "z_last_um":         z_last,
            "z_needed_um":       z_needed,
            "n_frames_current":  len(frame_table),
            "n_frames_trimmed":  n_trimmed,
            "n_frames_removed":  len(frame_table) - n_trimmed,
        })
    trim_df = pd.DataFrame(trim_rows)
    trim_df.to_csv(trim_cache, index=False)
    trim_meta_cache.write_text(json.dumps(trim_signature))
    print(f"Computed trim table for {len(trim_df)} FOV(s); cached to {trim_cache.name}.")

_capped = (trim_df["z_last_um"] + Z_MARGIN_UM) > Z_MAX_TRIMMED_UM
n_capped = int(_capped.sum())
if n_capped > 0:
    max_truncated_um = float((trim_df["z_last_um"] + Z_MARGIN_UM - Z_MAX_TRIMMED_UM).clip(lower=0).max())
    print(f"\nWARNING: Z_MAX_TRIMMED_UM={Z_MAX_TRIMMED_UM:.1f} um is BELOW z_last_um+Z_MARGIN_UM for "
          f"{n_capped}/{len(trim_df)} FOV(s) -- the trimmed scheme would cut off up to "
          f"{max_truncated_um:.1f} um of real measured tissue signal for those FOV(s). "
          f"Raise Z_MAX_TRIMMED_UM in the parameters cell above if this isn't intentional.")

In [ ]:
# ---- Display --------------------------------------------------------------
def format_bytes(n):
    n = float(n)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if abs(n) < 1024 or unit == "TiB":
            return f"{n:.2f} {unit}"
        n /= 1024


# config.frame_width/frame_height are only needed to reshape raw .dax bytes --
# .zarr/.tiff carry their own shape, so read one real frame directly instead.
_sample_fpath = next(f for f in files if f.exists())
_sample_frame = next(frame for _, frame in iter_image_frames(
    _sample_fpath, [0], frame_width=config.frame_width, frame_height=config.frame_height,
))
frame_height_px, frame_width_px = _sample_frame.shape
frame_bytes = frame_width_px * frame_height_px * 2   # uint16 -- 2 bytes/pixel

# Theoretical time per frame -- from this round's HAL <exposure_time>, the
# same convention acquisition.dave.estimate_dave_experiment uses. The only
# rate available here, since the bits rounds this is planning for haven't
# been imaged yet (no real file-write timing exists to measure).
exposure_time_s = None
for s in meta.series_for_round(target_round_id):
    if not s.hal_config:
        continue
    exp = read_hal_exposure_time(Path(config.settings_dir) / s.hal_config)
    if exp is not None:
        exposure_time_s = exp
        break
if exposure_time_s is None:
    exposure_time_s = 0.25
    print("WARNING: could not read <exposure_time> from this round's HAL config -- "
          "falling back to 0.25 s/frame (same fallback acquisition.dave.estimate_dave_experiment uses).")
theoretical_time_per_frame_s = exposure_time_s
print(f"Theoretical time per frame: {theoretical_time_per_frame_s:.4f} s/frame (from HAL exposure_time)")

n_fovs                          = len(trim_df)
bytes_saved_per_fov              = trim_df["n_frames_removed"] * frame_bytes
total_bytes_current_this_round   = n_fovs * len(frame_table) * frame_bytes
total_bytes_saved_this_round     = int(bytes_saved_per_fov.sum())

print(f"\n--- Round {target_round_id}, {n_fovs} FOV(s) ---")
print(f"Frames/FOV: {len(frame_table)} -> mean {trim_df['n_frames_trimmed'].mean():.1f} "
      f"({trim_df['n_frames_removed'].mean():.1f} removed/FOV on average)")
print(f"Space: {format_bytes(total_bytes_current_this_round)} -> "
      f"{format_bytes(total_bytes_current_this_round - total_bytes_saved_this_round)}  "
      f"(saved {format_bytes(total_bytes_saved_this_round)}, "
      f"{100 * total_bytes_saved_this_round / total_bytes_current_this_round:.1f}%)")

time_saved_per_fov_s = trim_df["n_frames_removed"] * theoretical_time_per_frame_s
total_time_current_s = n_fovs * len(frame_table) * theoretical_time_per_frame_s
total_time_saved_s   = float(time_saved_per_fov_s.sum())

print(f"\nTime (theoretical, {theoretical_time_per_frame_s:.4f} s/frame):")
print(f"  Round total:  {format_duration(total_time_current_s)} -> "
      f"{format_duration(total_time_current_s - total_time_saved_s)}  "
      f"(saved {format_duration(total_time_saved_s)}, "
      f"{100 * total_time_saved_s / total_time_current_s:.1f}%)")
print(f"  Single FOV file (average): {trim_df['n_frames_removed'].mean():.1f} frame(s) removed x "
      f"{theoretical_time_per_frame_s:.4f} s/frame = {format_duration(time_saved_per_fov_s.mean())} saved")

if N_ROUNDS_LIKE_THIS != 1:
    print(f"\n--- Extrapolated to {N_ROUNDS_LIKE_THIS} round(s) assumed to share this z-sweep "
          f"(N_ROUNDS_LIKE_THIS) ---")
    print(f"Space saved: {format_bytes(total_bytes_saved_this_round * N_ROUNDS_LIKE_THIS)}")
    print(f"Time saved: {format_duration(total_time_saved_s * N_ROUNDS_LIKE_THIS)}")

trim_csv = config.analysis_dir / f"tissue_thickness_zrange_trim_round{target_round_id}.csv"
trim_df.to_csv(trim_csv, index=False)
print(f"\nSaved: {trim_csv}")

## 9 — Export: per-FOV z table for notebook 05

The one thing `05_create_hal_config_and_shutters_multi_z.ipynb` actually needs from this notebook: each FOV's own required z depth (`z_needed_um` from section 8's trim table -- `z_last_um` plus `Z_MARGIN_UM`, capped at `Z_MAX_TRIMMED_UM`), saved to a fixed, predictable location under `metadata/` so notebook 05 doesn't need to know this notebook's own round-number-specific output filenames. Re-running this cell after re-running section 8 (e.g. with a different `THRESHOLD`) simply overwrites it -- there is only ever one "current" z-per-FOV table for the experiment.

In [ ]:
z_per_fov_table = trim_df[["fov_id", "z_needed_um"]].copy()
z_per_fov_table["z_needed_um"] = z_per_fov_table["z_needed_um"].round(1)

z_per_fov_path = config.metadata_dir / "z_per_fov_table.csv"
z_per_fov_table.to_csv(z_per_fov_path, index=False)

n_no_z_needed = int(z_per_fov_table["z_needed_um"].isna().sum())
print(f"Saved: {z_per_fov_path}")
print(f"{len(z_per_fov_table)} FOV(s); {n_no_z_needed} with no z_needed_um "
      f"(no detected signal at all in this FOV -- see section 6).")
print(z_per_fov_table["z_needed_um"].describe())